# GEPA prompt evolution for Adaptive-MPT on BBQ (Kaggle)

Runs `mpt_gepa_bbq.py`, which uses the official [`gepa`](https://github.com/gepa-ai/gepa) package. The fitness set has **K groups × 11 categories × 4 rows** from `data/BBQ` (each group covers neg/nonneg × ambig/disambig). None of these rows is in the **880-row `--truncate` evaluation set** (rebuilt from `TRUNCATE_SEED = 42`), so the final benchmark stays clean.

**Before running**
- Add this repo as a Kaggle Dataset. It must include `data/BBQ/*.jsonl`.
- Accelerator: GPU (T4 ×2 or P100). **Internet: ON** (needed for pip + OpenRouter).
- Kaggle Secrets: `OPENROUTER_API_KEY` (the reflector LLM), and `HF_TOKEN` for gated models such as Llama.

**Resume:** after a session timeout, re-run all cells with the same `OUT_DIR`. GEPA continues from `OUT_DIR/gepa`, and the epoch count carries over.

In [ ]:
import os, sys, subprocess
REPO = '/kaggle/input/adaptive-mpt'          # dataset containing this repo (read-only)
WORK = '/kaggle/working/adaptive-mpt'
if not os.path.exists(WORK):
    subprocess.run(['cp', '-r', REPO, WORK], check=True)
os.chdir(WORK); sys.path.insert(0, WORK)
!pip -q install -r requirements.txt          # includes gepa==0.1.4
!python -c "import gepa, vllm; print('gepa ok | vllm', vllm.__version__)"

In [ ]:
# Secrets: HF_TOKEN (gated models) and OPENROUTER_API_KEY (reflector)
from kaggle_secrets import UserSecretsClient
for name in ['HF_TOKEN', 'OPENROUTER_API_KEY']:
    try:
        os.environ[name] = UserSecretsClient().get_secret(name)
        print(f'{name}: set')
    except Exception as e:
        print(f'{name}: missing ({e})')
assert os.environ.get('OPENROUTER_API_KEY'), 'add the OPENROUTER_API_KEY secret (Add-ons -> Secrets)'
!nvidia-smi -L

## Config
- `K`: groups per category, so `K*11*4` rows. `VAL_PER_CAT` of them per category go to the GEPA valset (Pareto front); the rest go to the trainset.
- `EPOCHS`: passes over the trainset in minibatches of `MINIBATCH` groups. There are `ceil(train_groups / MINIBATCH)` GEPA iterations per epoch.
- `MAX_METRIC_CALLS`: an extra cap on rollouts (one rollout = one 4-row group through the pipeline). 0 means only `EPOCHS` limits the run.

In [ ]:
MODEL            = 'Qwen/Qwen2.5-7B-Instruct'
REFLECTOR        = 'nvidia/nemotron-3-ultra-550b-a55b:free'   # NVIDIA: Nemotron 3 Ultra (free) on OpenRouter
REFLECTOR_TOKENS = 16384    # thinking + answer; raise if the log says finish_reason=length
REFLECTOR_EFFORT = 'medium' # low | medium | high
K                = 3
VAL_PER_CAT      = 1
EPOCHS           = 3
MINIBATCH        = 3
MAX_METRIC_CALLS = 0
LAM              = 1.0      # group score = acc - LAM * bias
TRUNCATE_SEED    = 42       # --random_seed of the 880-row --truncate set to exclude
SEED             = 42
NUM_GPUS         = 2        # T4 x2 -> 2, P100 -> 1
MAX_MODEL_LEN    = 8192
APPLY_BEST       = True     # overwrite adaptive_mpt/prompts.py at the end (only if better on val; backup .bak)
OUT_DIR          = '/kaggle/working/evolved/gepa_bbq'

COMMON = (f'--k {K} --val_per_cat {VAL_PER_CAT} --truncate_seed {TRUNCATE_SEED} --seed {SEED} '
          f'--out_dir {OUT_DIR}')

In [ ]:
# 1. Data check only (no model): 880 excluded, 0 overlap, groups per category, iterations per epoch
!python mpt_gepa_bbq.py --dry_run {COMMON} --epochs {EPOCHS} --minibatch {MINIBATCH}

In [ ]:
# 2. GEPA run (resumable: re-run this cell with the same OUT_DIR)
flags = (f'{COMMON} --models {MODEL} --chat_templates --num_gpus {NUM_GPUS} --max_model_len {MAX_MODEL_LEN} '
         f'--epochs {EPOCHS} --minibatch {MINIBATCH} --max_metric_calls {MAX_METRIC_CALLS} --lam {LAM} '
         f'--reflector_model {REFLECTOR} --reflector_max_tokens {REFLECTOR_TOKENS} --reflector_effort {REFLECTOR_EFFORT}' + (' --apply_best' if APPLY_BEST else ''))
print(flags)
!python mpt_gepa_bbq.py {flags}

In [ ]:
# 3. Results: validation score of every candidate (candidate 0 = original prompts)
import json
import matplotlib.pyplot as plt
summary = json.load(open(f'{OUT_DIR}/summary.json'))
print({k: v for k, v in summary.items() if k != 'val_scores'})
s = summary['val_scores']
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(len(s)), s, color=['#888'] + ['#4c72b0'] * (len(s) - 1))
ax.bar(summary['best_idx'], s[summary['best_idx']], color='#dd8452', label='best')
ax.set_xlabel('candidate'); ax.set_ylabel('val score (acc - lam*bias)'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/val_scores.png', dpi=130); plt.show()

In [ ]:
# 4. What GEPA changed: diff of each evolved prompt against the original
import difflib, importlib
if os.path.exists('adaptive_mpt/prompts.py.bak'):
    src = open('adaptive_mpt/prompts.py.bak').read()        # prompts.py was already overwritten
else:
    src = open('adaptive_mpt/prompts.py').read()
orig_ns = {}; exec(src, orig_ns)
best = json.load(open(f'{OUT_DIR}/best_prompts.json'))
for name, text in best.items():
    if text == orig_ns[name]:
        print(f'= {name}: unchanged'); continue
    print(f'~ {name}:')
    for line in difflib.unified_diff(orig_ns[name].splitlines(), text.splitlines(), 'original', 'evolved', lineterm=''):
        print('   ', line)

In [ ]:
# 5. Save the outputs to /kaggle/working (download from the Output tab)
import shutil
if os.path.exists('adaptive_mpt/prompts.py.bak'):
    shutil.copy('adaptive_mpt/prompts.py', '/kaggle/working/prompts_evolved.py')
    print('prompts.py overwritten -> /kaggle/working/prompts_evolved.py (copy over adaptive_mpt/prompts.py in your repo)')
else:
    print('prompts.py not overwritten; apply later with:  python mpt_gepa_bbq.py --apply_only', f'{OUT_DIR}/best_prompts.json')
shutil.make_archive('/kaggle/working/gepa_bbq_run', 'zip', OUT_DIR)
print('run folder zipped -> /kaggle/working/gepa_bbq_run.zip')

## (Optional) Benchmark on the 880-row truncated set
Runs the full pipeline with the **current** `adaptive_mpt/prompts.py` (the evolved prompts if step 2 applied them) on the same 880 rows that were excluded from evolution. Then it evaluates them. Set `RUN_880 = True` to run it.

In [ ]:
RUN_880 = False
if RUN_880:
    !python mpt_inference.py --method mpt --note gepa --models {MODEL} --dataset BBQ --truncate \
        --random_seed {TRUNCATE_SEED} --num_gpus {NUM_GPUS} --max_model_len {MAX_MODEL_LEN} --chat_templates --overwrite
    !python mpt_evaluate.py --method mpt --note gepa --models {MODEL} --dataset BBQ --truncate